# Run RAG Query

This notebook is to run the RAG query. 

In [ ]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import ChatOllama

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")


# Reload Vector DB 

In [ ]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

In [ ]:
save_path = "fixed_rag_faiss_index"

vector_db = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True
)

print("Vector DB reloaded from local folder.")

In [ ]:
# Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

In [ ]:
# Optional terminal test:
#     ollama run llama3.1:8b

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.2
)

print("Ollama LLM initialized.")


In [ ]:
def call_llm(prompt: str) -> str:
    response = llm.invoke(prompt)

    # ChatOllama returns an AIMessage with a .content field
    if hasattr(response, "content"):
        return response.content

    return str(response)

print("LLM helper ready.")

In [ ]:
def retrieve_chunks(query: str, k: int = 3):
    """
    Retrieve top-k chunks with their citation metadata.
    """
    results = vector_db.similarity_search(query, k=k)
    return results


# Test it
test_chunks = retrieve_chunks("What is copyright?", k=5)
print(f"Retrieved {len(test_chunks)} chunks")

for i, doc in enumerate(test_chunks, start=1):
    print(f"\nChunk {i}")
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page"))
    print("Citation:", doc.metadata.get("ieee_citation"))

In [ ]:
def rag_query(question: str, k: int = 4) -> dict:
    """
    Complete RAG pipeline:
    1. Retrieve chunks
    2. Include full agent-generated IEEE citation metadata
    3. Generate answer locally with Ollama
    4. Return answer and references
    """
    try:
        retrieved_docs = retrieve_chunks(question, k)

        context_blocks = []

        for i, doc in enumerate(retrieved_docs, start=1):
            source_file = doc.metadata.get("source_file", "Unknown file")
            page = doc.metadata.get("page", "Unknown page")
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            context_blocks.append(
                f"""
Source [{i}]
File: {source_file}
Page: {page}
Full IEEE Citation: {citation}

Content:
{doc.page_content}
"""
            )

        context = "\n\n---\n\n".join(context_blocks)

        prompt = f"""
Answer the question using ONLY the retrieved PDF content below.

Rules:
- Use IEEE-style inline citations like [1], [2], etc.
- Cite the specific source number that supports each claim.
- When helpful, include page numbers in the sentence, like [1, p. 3].
- Do not cite sources that do not support the answer.
- If the retrieved PDFs do not contain enough information, say that clearly.
- At the end, include a References section.
- The References section must include the full IEEE citation for every source cited.

Retrieved PDF Content:
{context}

Question:
{question}

Answer:
"""

        response = call_llm(prompt)

        references = []
        seen = set()

        for i, doc in enumerate(retrieved_docs, start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            if citation not in seen:
                references.append(f"[{i}] {citation}")
                seen.add(citation)

        return {
            "question": question,
            "answer": response,
            "references": references,
            "retrieved_docs": retrieved_docs
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"error": str(e)}

In [ ]:
# Tests
result = rag_query("What is the impact of LLMs on copyright?")

if "error" in result:
    print("RAG failed:", result["error"])
else:
    print(f"Question: {result['question']}")
    print(f"Answer: {result['answer']}")